# Coupled non-CSS surface-code decoding

Decode the full coupled q/p lattice rather than splitting the code into CSS sectors.

In [ ]:
import Pkg
repo_root = isfile(joinpath(pwd(), "Project.toml")) ? pwd() : normpath(joinpath(pwd(), ".."))
Pkg.activate(repo_root)

using LinearAlgebra
using LatticeDecoder
using Random

samples_per_point = parse(Int, get(ENV, "LATTICEDECODER_EXAMPLE_SAMPLES", "20"))

In [ ]:
d = 3
code_definition = GKP_Surface_Code(d, false)
M = code_definition.code
J = code_definition.J
H = -M * J
G = J * inv(M)
logical_check = inv(M)
problem = QuantumDecodingProblem(H, G, logical_check)

sigma = 0.12
rng = MersenneTwister(40)
error_vector = sample_error(rng, sigma, size(H, 2))
received = copy(error_vector)
decoder = LDLCDecoder(
    initialize_tanner_graph(H);
    schedule=:serial,
    algorithm=:lsd,
    sigma,
    max_iterations=size(H, 2),
)
soft_estimate = run_decoder!(decoder, received)
decision = hard_decision(soft_estimate, H)
residual = error_vector - (received - G * decision)

(logical_error=is_logical_error(logical_check, residual), decision_norm=norm(decision))

In [ ]:
sigmas = [0.08, 0.12, 0.16]
estimates = [
    estimate_logical_error_rate!(
        MersenneTwister(400),
        LDLCDecoder(
            initialize_tanner_graph(H);
            schedule=:serial,
            algorithm=:lsd,
            sigma,
            max_iterations=size(H, 2),
        ),
        problem;
        samples=samples_per_point,
    )
    for sigma in sigmas
]

[
    (
        sigma=sigma,
        failures=result.events,
        samples=result.samples,
        rate=result.rate,
        interval=(result.lower, result.upper),
    )
    for (sigma, result) in zip(sigmas, estimates)
]

The failure test checks whether the residual has any nonintegral logical coordinate. Report raw failures as well as the estimated rate.